# Trend Tracker — Notebook 01: Token Preprocess

SQL extract → clean, filtered token corpus. Pure wrangling — no analytical work.

| Step | What | Key output |
|------|------|------------|
| 1 | Ingest | `prepared/01_ingested.parquet` |
| 2 | Token normalization | `prepared/03_lemmatized.parquet` |
| 3+4 | Consolidation (auto-build + apply) | `prepared/04_consolidated.parquet`, `CONFIG/consolidation_map.csv` |
| 5 | Sparsity filter | `prepared/05_filtered.parquet`, `vocab/unigram_*.csv` |
| 6 | Quality CP1 | `quality/quality_cp1.json` |


## Setup

In [1]:
import sys, re
from pathlib import Path
import pandas as pd
import numpy as np
import simplemma

ROOT = Path(".")
sys.path.insert(0, str(ROOT))
from utils import (
    load_cfg, ingest, flat_freq, token_doc_freq, quality_report,
    build_output_path, ensure_warning_file, append_warning,
    start_stage_manifest, finalize_stage_manifest, artifact_meta
)

CFG = load_cfg()

def OUT(subdir, fname):
    return build_output_path(subdir, fname)

WARNINGS_PATH = OUT("prepared", "warnings_01.jsonl")
ensure_warning_file(WARNINGS_PATH)

STAGE_MANIFEST = start_stage_manifest(
    stage_name="01_preprocess",
    notebook_file="01_token_preprocess_v0.1.ipynb",
    run_id=None,
    group_by_field=None,
    filter_fields_key=None,
)

print("Config:", CFG)
print(f"Warnings path: {WARNINGS_PATH}")

Config: {'sql': {'min_doc_count': 15, 'max_doc_fraction': 0.5, 'token_cap': 80}, 'preprocess': {'min_project_tokens': 5, 'dedup_tokens_per_project': True}, 'ingest': {'chunk_size': 500000}, 'ngrams': {'max_n': 3}, 'tfidf': {'min_df': 7, 'max_df': 0.6, 'ngram_range': [1, 2]}, 'nmf': {'n_components': 12, 'top_words': 12, 'random_seed': 42, 'max_iter': 400}, 'enrichment': {'rare_doc_freq_ceiling': 1000, 'agg_distance_threshold': 0.35, 'min_cluster_size': 3, 'min_inject_projects': 25, 'max_doc_freq_for_injection': 25000, 'classify_batch_size': 150, 'embedding_model': 'all-MiniLM-L6-v2', 'cache_schema_version': 'v1', 'coverage_deviation_warn_threshold': 10, 'force_reclassify': True}, 'analysis': {'group_by': 'project_category', 'min_group_projects': 50, 'min_coverage': 6, 'topic_assignment_threshold': 0.3, 'nmf_min_shared': 3, 'cat_tfidf_top_n': 20, 'review_group': None, 'exclude_groups': [], 'synthesis_max_workers': 8, 'labeling_max_workers': 8, 'bins': [], 'filter_logic': 'and', 'filters'

---
## Step 1 — Ingest

`tokens` arrives as a comma-separated LISTAGG string from SQL; `ingest()`
splits it into a list. `theme_version` is planted here for Phase 2+ registry
continuity.


In [2]:
DATA_DIR   = ROOT / "DATA"
CHUNK_SIZE = CFG["ingest"]["chunk_size"]

# ── 1. Union all project_essay* CSVs via chunked reads ───────────────────────
essay_files = sorted(DATA_DIR.glob("project_essay*.csv"))
if not essay_files:
    raise FileNotFoundError(f"No project_essay*.csv files found in {DATA_DIR}")
print(f"Found {len(essay_files)} essay file(s): {[f.name for f in essay_files]}")

ESSAY_COLS = ["project_id", "tokens", "posted_date", "funded_date"]
ESSAY_TEXT_CANDIDATES = ["essay", "essay_text", "full_text", "project_essay", "text"]

chunks = []
essay_lookup_chunks = []

for fpath in essay_files:
    all_cols = pd.read_csv(fpath, nrows=0).columns.tolist()
    use = [c for c in ESSAY_COLS if c in all_cols]

    text_col = next((c for c in ESSAY_TEXT_CANDIDATES if c in all_cols), None)
    if text_col and text_col not in use:
        use = use + [text_col]

    date_cols = [c for c in use if c in {"posted_date", "funded_date"}]

    for chunk in pd.read_csv(
        fpath,
        chunksize=CHUNK_SIZE,
        usecols=use,
        parse_dates=date_cols or False,
    ):
        if "tokens" in chunk.columns:
            essay_chunk = (
                chunk[["project_id", "tokens"]]
                .rename(columns={"tokens": "essay_text"})
                .copy()
            )
            essay_chunk["essay_text"] = (
                essay_chunk["essay_text"]
                .fillna("")
                .astype(str)
                .str.replace(",", " ", regex=False)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )
            essay_chunk = essay_chunk[essay_chunk["essay_text"] != ""]
            if not essay_chunk.empty:
                essay_lookup_chunks.append(essay_chunk)

        chunk["tokens"] = (
            chunk["tokens"].fillna("").str.split(",")
            .apply(lambda ts: [t.strip() for t in ts if t.strip()])
        )
        if text_col and text_col in chunk.columns:
            essay_chunk = (
                chunk[["project_id", text_col]]
                .rename(columns={text_col: "essay_text"})
                .copy()
            )
            essay_chunk["essay_text"] = (
                essay_chunk["essay_text"]
                .fillna("")
                .astype(str)
                .str.replace(r"\s+", " ", regex=True)
                .str.strip()
            )
            essay_chunk = essay_chunk[essay_chunk["essay_text"] != ""]
            if not essay_chunk.empty:
                essay_lookup_chunks.append(essay_chunk)
            chunk = chunk.drop(columns=[text_col])
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"  Raw union: {len(df):,} rows")

essay_lookup_df = (
    pd.concat(essay_lookup_chunks, ignore_index=True)
    if essay_lookup_chunks
    else pd.DataFrame(columns=["project_id", "essay_text"])
)

if not essay_lookup_df.empty:
    essay_lookup_df["essay_len"] = essay_lookup_df["essay_text"].str.len()
    essay_lookup_df = (
        essay_lookup_df
        .sort_values(["project_id", "essay_len"], ascending=[True, False])
        .drop_duplicates(subset=["project_id"], keep="first")
        .drop(columns=["essay_len"])
        .reset_index(drop=True)
    )

essay_lookup_path = OUT("prepared", "02_project_essay_lookup.parquet")
essay_lookup_df.to_parquet(essay_lookup_path, index=False)
print(f"  Essay lookup rows: {len(essay_lookup_df):,}")
print(f"  Saved essay lookup → {essay_lookup_path}")

# ── 2. Left-join project attributes ──────────────────────────────────────────
ATTR_CSV = DATA_DIR / "project_attributes.csv"
if not ATTR_CSV.exists():
    raise FileNotFoundError(f"project_attributes.csv not found in {DATA_DIR}")

attrs = pd.read_csv(ATTR_CSV)
df    = df.merge(attrs, on="project_id", how="left")
df["posted_date"] = pd.to_datetime(df["posted_date"])
df["posted_year"] = df["posted_date"].dt.year
df["posted_year_quarter"] = (
    df["posted_date"].dt.year.astype(str)
    + "_Q"
    + df["posted_date"].dt.quarter.astype(str)
)
print(f"  After attribute join: {len(df):,} rows  |  columns: {list(df.columns)}")

df["theme_version"] = "1.0"
df.to_parquet(OUT("prepared", "01_ingested.parquet"), index=False)
print(f"\nSaved 01_ingested.parquet")

Found 14 essay file(s): ['project_essays_01.csv', 'project_essays_02.csv', 'project_essays_03.csv', 'project_essays_04.csv', 'project_essays_05.csv', 'project_essays_06.csv', 'project_essays_07.csv', 'project_essays_08.csv', 'project_essays_09.csv', 'project_essays_10.csv', 'project_essays_11.csv', 'project_essays_12.csv', 'project_essays_13.csv', 'project_essays_14.csv']
essay_lookup_chunks count: 14
first chunk shape: (89080, 2)
first chunk columns: ['project_id', 'essay_text']
first chunk dtypes:
project_id     int64
essay_text    object
dtype: object
first chunk non-empty essay_text rows: 89080
sample essay_text value (first row):
'past opportunity assistance physical therapist class concerned uncomfortable seats students sit portion grade students struggling learners students attention deficit problems school serves majority fa'

Columns read from first file: ['project_id', 'tokens']

Raw tokens column sample (first 3 rows):
  project_id=1243052  tokens='past,opportunity,assistanc

---
## Step 2 — Token Normalization

Uses `simplemma` for morphological normalization — handles inflected forms
(`drives → drive`, `organized → organize`, `lives → live`) correctly.
Domain terms and proper nouns it does not recognise are returned unchanged;
the consolidation map handles any residual cases.
Also update the table in cell 0: output filename stays `03_lemmatized.parquet`.

In [3]:
def normalize_tokens(tokens):
    """Light morphological normalization via simplemma."""
    return [simplemma.lemmatize(t, lang="en") for t in tokens]

df["tokens"] = df["tokens"].apply(normalize_tokens)
df.to_parquet(OUT("prepared", "03_lemmatized.parquet"), index=False)
print(f"Saved 03_lemmatized.parquet  |  {len(df):,} rows")


Saved 03_lemmatized.parquet  |  1,255,721 rows


---
## Steps 3 + 4 — Consolidation: Auto-build & Apply

Builds a consolidation map fully automatically — no human review step.

**Two automatic passes:**
1. **Known-fix dictionary** — corrects documented simplemma truncation artifacts
   (e.g. `creat → create`, `organiz → organize`). Extend `KNOWN_FIXES` to add
   any domain-specific corrections over time.
2. **Plural-pair detection** — within the top-N vocab, maps `word + s/es → word`
   only when the singular form also appears in the same vocab window.

Writes `CONFIG/consolidation_map.csv` (for audit trail) and
`OUTPUTS/prepared/consolidation_candidates.csv` (all top-N tokens with auto-filled replacements).
Then applies the map to `df` in-place.


In [4]:
_TRUNC_RE = re.compile(r"(at|iv|iz|az|ig|if|ic|olog)$")

# ── Known simplemma truncation fixes ─────────────────────────────────────────
# Extend this dict if you find new bad stems in the quality report.
KNOWN_FIXES = {
    "creat":     "create",
    "organiz":   "organize",
    "writ":      "write",
    "engag":     "engage",
    "inspir":    "inspire",
    "challeng":  "challenge",
    "experienc": "experience",
    "includ":    "include",
    "improv":    "improve",
    "achiev":    "achieve",
    "contribut": "contribute",
    "encourag":  "encourage",
    "provid":    "provide",
    "requir":    "require",
    "recogniz":  "recognize",
    "utiliz":    "utilize",
    "analyz":    "analyze",
    "communic":  "communicate",
    "particip":  "participate",
    "collabor":  "collaborate",
    "introduc":  "introduce",
    "practic":   "practice",
    "explor":    "explore",
    "prepar":    "prepare",
    "develop":   "develop",
}

def _auto_replacement(original: str, token_set: set) -> str:
    """Return replacement string, or '' to leave token unchanged."""
    if original in KNOWN_FIXES:
        return KNOWN_FIXES[original]
    if original.endswith("s") and not original.endswith("ss") and len(original) > 3:
        singular = original[:-1]
        if singular in token_set:
            return singular
    if original.endswith("es") and len(original) > 4:
        singular = original[:-2]
        if singular in token_set:
            return singular
    return ""

def build_consolidation_candidates(df, top_n=1000):
    vocab = flat_freq(df).head(top_n).reset_index()
    vocab.columns = ["original", "freq"]
    vocab["flag_type"] = vocab["original"].apply(
        lambda w: "truncation" if len(w) <= 8 and _TRUNC_RE.search(w) else ""
    )
    token_set = set(vocab["original"])
    vocab["replacement"] = vocab["original"].apply(
        lambda w: _auto_replacement(w, token_set)
    )
    vocab["notes"] = "auto"
    return vocab

# Build map
candidates = build_consolidation_candidates(df)
MAP_PATH = ROOT / "CONFIG/consolidation_map.csv"
MAP_PATH.parent.mkdir(parents=True, exist_ok=True)
candidates.to_csv(MAP_PATH, index=False)
print(f"{len(candidates)} tokens scanned  |  "
      f"{(candidates.replacement != '').sum()} auto-mapped  |  "
      f"{(candidates.flag_type == 'truncation').sum()} truncation-flagged")

# Apply map
cmap = candidates[candidates["replacement"] != ""].copy()
mapping = dict(zip(cmap["original"].str.strip(), cmap["replacement"].str.strip()))
df["tokens"] = df["tokens"].apply(lambda ts: [mapping.get(t, t) for t in ts])
cmap[["original", "replacement", "flag_type", "notes"]].to_csv(
    OUT("prepared", "consolidation_audit.csv"), index=False
)
df.to_parquet(OUT("prepared", "04_consolidated.parquet"), index=False)

WATCH = {"creat","engag","experienc","organiz","includ","writ","inspir","challeng"}
found = flat_freq(df)[lambda s: s.index.isin(WATCH)]
print(f"Mappings applied: {len(mapping)}")
print("✓ No watched stems remain" if found.empty
      else f"⚠ Still present:\n{found.to_string()}")
candidates[candidates["replacement"] != ""].head(20)


1000 tokens scanned  |  6 auto-mapped  |  17 truncation-flagged
Mappings applied: 6
✓ No watched stems remain


,original,freq,flag_type,replacement,notes
93,develop,194450,,develop,auto
287,ipads,76303,,ipad,auto
315,means,71559,,mean,auto
611,towards,35794,,toward,auto
718,bags,29575,,bag,auto
850,whiteboards,24427,,whiteboard,auto


---
## Step 5 — Sparsity Filter

Applies corpus-level document-frequency bounds from `params.yaml`. Projects
below `min_project_tokens` are written to a separate exclusion artifact and
removed from the main corpus. `excluded_flag` is not carried on retained rows —
by construction every retained row passed the filter.

Deduplication (from `params.yaml`): each token counted once per project, making
TF-IDF reflect document presence rather than within-project repetition.


In [5]:
sq, sp_cfg = CFG["sql"], CFG["preprocess"]

doc_freq = token_doc_freq(df)
keep     = doc_freq[(doc_freq >= sq["min_doc_count"]) &
                    (doc_freq <  len(df) * sq["max_doc_fraction"])].index

dedup = sp_cfg["dedup_tokens_per_project"]
df["tokens"] = df["tokens"].apply(
    lambda ts: list(dict.fromkeys(t for t in ts if t in keep))
    if dedup else [t for t in ts if t in keep]
)

min_tok  = sp_cfg["min_project_tokens"]
excluded = df[df["tokens"].apply(len) < min_tok].copy()
excluded["exclusion_reason"] = f"< {min_tok} tokens after filtering"
df       = df[df["tokens"].apply(len) >= min_tok].reset_index(drop=True)

# Enforce per-project token cap from sql.token_cap
token_cap = CFG["sql"]["token_cap"]
df["tokens"] = df["tokens"].apply(lambda ts: ts[:token_cap])

doc_freq.loc[keep].to_csv(OUT("vocab", "unigram_docfreq.csv"), header=True)
flat_freq(df).to_csv(OUT("vocab", "unigram_freq.csv"), header=True)
excluded.to_parquet(OUT("prepared", "excluded_projects.parquet"), index=False)
df.to_parquet(OUT("prepared", "05_filtered.parquet"), index=False)

print(f"Retained: {len(df):,}  |  Excluded: {len(excluded):,}  |  Vocab: {len(keep):,}")


Retained: 1,255,697  |  Excluded: 24  |  Vocab: 7,392


---
## Step 6 — Quality Report: Checkpoint 1

SQL refinement loop exit point. Review the quality report and top vocabulary
before proceeding to analysis.


In [6]:
qr1 = quality_report(df, label="cp1", doc_freq=doc_freq.loc[keep],
                     save_path=OUT("quality", "quality_cp1.json"))

stage_manifest_path = OUT("prepared/metadata", "stage_manifest_01_preprocess.json")
finalize_stage_manifest(
    STAGE_MANIFEST,
    output_path=stage_manifest_path,
    status="success",
    input_artifacts=[
        artifact_meta(DATA_DIR / "project_attributes.csv", "project_attributes_csv"),
        *[artifact_meta(p, f"essay_source::{p.name}") for p in essay_files],
    ],
    output_artifacts=[
        artifact_meta(OUT("prepared", "01_ingested.parquet"), "ingested_parquet"),
        artifact_meta(OUT("prepared", "02_project_essay_lookup.parquet"), "project_essay_lookup_parquet"),
        artifact_meta(OUT("prepared", "03_lemmatized.parquet"), "lemmatized_parquet"),
        artifact_meta(OUT("prepared", "04_consolidated.parquet"), "consolidated_parquet"),
        artifact_meta(OUT("prepared", "05_filtered.parquet"), "filtered_parquet"),
        artifact_meta(OUT("prepared", "excluded_projects.parquet"), "excluded_projects_parquet"),
        artifact_meta(OUT("vocab", "unigram_docfreq.csv"), "unigram_docfreq_csv"),
        artifact_meta(OUT("vocab", "unigram_freq.csv"), "unigram_freq_csv"),
        artifact_meta(OUT("quality", "quality_cp1.json"), "quality_cp1_json"),
    ],
    row_counts={
        "input_projects": int(len(df) + len(excluded)),
        "filtered_projects": int(len(df)),
        "excluded_projects": int(len(excluded)),
        "retained_vocab": int(len(keep)),
    },
    key_params={
        "min_doc_count": CFG["sql"]["min_doc_count"],
        "max_doc_fraction": CFG["sql"]["max_doc_fraction"],
        "min_project_tokens": CFG["preprocess"]["min_project_tokens"],
        "token_cap": CFG["sql"]["token_cap"],
    },
    warnings_path=WARNINGS_PATH,
)
print(f"Stage manifest written → {stage_manifest_path}")


=======================================================  [cp1]
  Projects : 1,255,697
  Tok/proj : min=5  p50=61  max=80
  Vocab    : 7,392 unique tokens
  Stopwords: FAIL — ['project', 'grade', 'teacher', 'education']

Stage manifest written → /Users/matt.fritz/Desktop/Research Insights/Essay Prototype/OUTPUTS/prepared/metadata/stage_manifest_01_preprocess.json
